# EXP-035 | Random Forest 앙상블

지금까지 LGB/CAT/XGB 모두 **Boosting** 계열 → 예측 상관관계 0.96+ 으로 다양성 제한.
RF는 **Bagging** 기반으로 inductive bias가 근본적으로 달라 상관관계 0.85~0.90 수준 예상.

| 전략 | 내용 |
|---|---|
| RF 단독 | 5-Fold OOF AUC 측정 |
| 상관관계 분석 | 기존 GBDT 예측과 상관관계 확인 |
| 앙상블 | RF + EXP-020 + EXP-028 + EXP-032 Rank Average |

| 기준선 | EXP-032 제출 0.74219 (현재 최고) |

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import warnings
from pathlib import Path
from datetime import date
from scipy.stats import rankdata

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, f1_score, recall_score, precision_score, accuracy_score
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier

from src.preprocessing import preprocess

warnings.filterwarnings('ignore')

DATA_DIR = Path('../data/raw')
SUB_DIR  = Path('../data/submissions')
DOCS_DIR = Path('../docs')
TARGET   = '임신 성공 여부'
SEED     = 42
N_FOLDS  = 5
EXP_NO   = 35
AUTHOR   = '조여진'
CV_STR   = f'Stratified {N_FOLDS}-Fold'

train = pd.read_csv(DATA_DIR / 'train.csv')
test  = pd.read_csv(DATA_DIR / 'test.csv')
sub   = pd.read_csv(DATA_DIR / 'sample_submission.csv')
print(f'train: {train.shape}  /  test: {test.shape}')

train: (256351, 69)  /  test: (90067, 68)


## 1. 피처 준비 (FE-v1)

In [2]:
def add_pre_encode_features(df):
    df = df.copy()
    df['기증_난자_여부'] = (df['난자 출처'] == '기증 제공').astype(int)
    df['기증_정자_여부'] = df['정자 출처'].isin(['기증 제공', '배우자 및 기증 제공']).astype(int)
    return df

def add_derived_features_v1(df):
    df = df.copy()
    eps = 1e-6
    df['수정률']    = df['총 생성 배아 수']   / (df['혼합된 난자 수'] + eps)
    df['이식률']    = df['이식된 배아 수']    / (df['총 생성 배아 수'] + eps)
    df['저장률']    = df['저장된 배아 수']    / (df['총 생성 배아 수'] + eps)
    df['ICSI_비율'] = df['미세주입된 난자 수'] / (df['혼합된 난자 수'] + eps)
    df['배아_발달일']    = df['배아 이식 경과일'] - df['난자 혼합 경과일']
    df['신선_시술_여부']  = df['수집된 신선 난자 수'].notna().astype(int)
    male_cols   = ['남성 주 불임 원인','남성 부 불임 원인','불임 원인 - 남성 요인']
    female_cols = ['여성 주 불임 원인','여성 부 불임 원인','불임 원인 - 난관 질환',
                   '불임 원인 - 배란 장애','불임 원인 - 자궁내막증','불임 원인 - 자궁경부 문제']
    couple_cols = ['부부 주 불임 원인','부부 부 불임 원인']
    sperm_cols  = ['불임 원인 - 정자 농도','불임 원인 - 정자 운동성',
                   '불임 원인 - 정자 형태','불임 원인 - 정자 면역학적 요인']
    all_cause   = male_cols + female_cols + couple_cols + sperm_cols + ['불명확 불임 원인']
    df['남성_불임_합계']      = df[male_cols].sum(axis=1)
    df['여성_불임_합계']      = df[female_cols].sum(axis=1)
    df['부부_불임_합계']      = df[couple_cols].sum(axis=1)
    df['정자_문제_합계']      = df[sperm_cols].sum(axis=1)
    df['총_불임원인_수']      = df[all_cause].sum(axis=1)
    df['임신시도기록_있음']    = df['임신 시도 또는 마지막 임신 경과 연수'].notna().astype(int)
    df['신선_난자_저장_있음']  = (df['저장된 신선 난자 수'] > 0).astype(int)
    df['나이_시술횟수_상호작용'] = df['시술 당시 나이'] * df['총 시술 횟수']
    return df

train_fe = add_pre_encode_features(train)
test_fe  = add_pre_encode_features(test)
X_base_tr, X_base_te = preprocess(train_fe, test_fe)
X_train = add_derived_features_v1(X_base_tr)
X_test  = add_derived_features_v1(X_base_te)

# RF/ET는 NaN 처리 필요 (GBDT와 달리 NaN 미지원)
col_medians = X_train.median()
X_train_filled = X_train.fillna(col_medians)
X_test_filled  = X_test.fillna(col_medians)

y_train = train[TARGET]
print(f'X_train: {X_train.shape}  /  NaN 채운 후 NaN 수: {X_train_filled.isna().sum().sum()}')

X_train: (256351, 85)  /  NaN 채운 후 NaN 수: 0


## 2. Random Forest + ExtraTrees 학습

In [3]:
RF_PARAMS = dict(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=20,
    max_features='sqrt',
    class_weight='balanced',
    n_jobs=-1,
    random_state=SEED,
)
ET_PARAMS = dict(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=20,
    max_features='sqrt',
    class_weight='balanced',
    n_jobs=-1,
    random_state=SEED,
)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

oof_rf  = np.zeros(len(X_train))
oof_et  = np.zeros(len(X_train))
test_rf = np.zeros(len(X_test))
test_et = np.zeros(len(X_test))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train_filled, y_train), 1):
    X_tr  = X_train_filled.iloc[tr_idx]
    X_val = X_train_filled.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

    rf = RandomForestClassifier(**RF_PARAMS)
    rf.fit(X_tr, y_tr)
    oof_rf[val_idx] = rf.predict_proba(X_val)[:, 1]
    test_rf        += rf.predict_proba(X_test_filled)[:, 1] / N_FOLDS

    et = ExtraTreesClassifier(**ET_PARAMS)
    et.fit(X_tr, y_tr)
    oof_et[val_idx] = et.predict_proba(X_val)[:, 1]
    test_et        += et.predict_proba(X_test_filled)[:, 1] / N_FOLDS

    print(f'  Fold {fold}  RF={roc_auc_score(y_val, oof_rf[val_idx]):.5f}  '
          f'ET={roc_auc_score(y_val, oof_et[val_idx]):.5f}')

auc_rf = roc_auc_score(y_train, oof_rf)
auc_et = roc_auc_score(y_train, oof_et)
auc_rf_et = roc_auc_score(y_train,
    np.stack([rankdata(oof_rf), rankdata(oof_et)], axis=1).mean(axis=1))
print(f'\n[RF]  OOF AUC: {auc_rf:.5f}')
print(f'[ET]  OOF AUC: {auc_et:.5f}')
print(f'[RF+ET Rank Avg] OOF AUC: {auc_rf_et:.5f}')

  Fold 1  RF=0.73395  ET=0.73372
  Fold 2  RF=0.73869  ET=0.73838
  Fold 3  RF=0.73731  ET=0.73671
  Fold 4  RF=0.73476  ET=0.73471
  Fold 5  RF=0.73690  ET=0.73533

[RF]  OOF AUC: 0.73631
[ET]  OOF AUC: 0.73575
[RF+ET Rank Avg] OOF AUC: 0.73672


## 3. 기존 GBDT 예측값과 상관관계 분석

In [4]:
# 기존 submission 파일 로드 (test 예측값)
gbdt_files = {
    'EXP-020': SUB_DIR / 'submission_exp020_조여진_07407_normalized.csv',
    'EXP-028': SUB_DIR / 'submission_exp028_조여진_07408.csv',
    'EXP-032': SUB_DIR / 'submission_exp032_조여진_07407.csv',
}
gbdt_preds = {name: pd.read_csv(path)['probability'].values
              for name, path in gbdt_files.items()}

# RF+ET 앙상블 test 예측
test_rf_et = np.stack([rankdata(test_rf), rankdata(test_et)], axis=1).mean(axis=1)

all_preds = {**gbdt_preds, 'RF': test_rf, 'ET': test_et, 'RF+ET': test_rf_et}
corr_df = pd.DataFrame(all_preds).corr()

print('예측값 상관관계 (낮을수록 앙상블 효과 큼):')
print(corr_df.round(4).to_string())
print()
print('▶ RF/ET ↔ GBDT 상관관계 요약:')
for gbdt_name in ['EXP-020', 'EXP-028', 'EXP-032']:
    print(f'  RF ↔ {gbdt_name}: {corr_df.loc["RF", gbdt_name]:.4f}  '
          f'ET ↔ {gbdt_name}: {corr_df.loc["ET", gbdt_name]:.4f}')

예측값 상관관계 (낮을수록 앙상블 효과 큼):
         EXP-020  EXP-028  EXP-032      RF      ET   RF+ET
EXP-020   1.0000   0.9611   0.9996  0.9470  0.9348  0.9833
EXP-028   0.9611   1.0000   0.9609  0.9884  0.9841  0.9493
EXP-032   0.9996   0.9609   1.0000  0.9467  0.9347  0.9831
RF        0.9470   0.9884   0.9467  1.0000  0.9919  0.9578
ET        0.9348   0.9841   0.9347  0.9919  1.0000  0.9483
RF+ET     0.9833   0.9493   0.9831  0.9578  0.9483  1.0000

▶ RF/ET ↔ GBDT 상관관계 요약:
  RF ↔ EXP-020: 0.9470  ET ↔ EXP-020: 0.9348
  RF ↔ EXP-028: 0.9884  ET ↔ EXP-028: 0.9841
  RF ↔ EXP-032: 0.9467  ET ↔ EXP-032: 0.9347


## 4. 앙상블 조합 비교

In [5]:
def rank_avg_norm(arrays):
    ranks = np.stack([rankdata(a) for a in arrays], axis=1)
    avg   = ranks.mean(axis=1)
    return (avg - avg.min()) / (avg.max() - avg.min())

p020 = gbdt_preds['EXP-020']
p028 = gbdt_preds['EXP-028']
p032 = gbdt_preds['EXP-032']

combos = {
    'RF만':                rank_avg_norm([test_rf]),
    'RF+ET':               rank_avg_norm([test_rf, test_et]),
    '020+028+032 (기존)':  rank_avg_norm([p020, p028, p032]),
    '020+028+032+RF':      rank_avg_norm([p020, p028, p032, test_rf]),
    '020+028+032+ET':      rank_avg_norm([p020, p028, p032, test_et]),
    '020+028+032+RF+ET':   rank_avg_norm([p020, p028, p032, test_rf, test_et]),
    '020+032+RF':          rank_avg_norm([p020, p032, test_rf]),
    '028+032+RF':          rank_avg_norm([p028, p032, test_rf]),
}

print(f'RF  OOF AUC: {auc_rf:.5f}  (참고: GBDT 기준선 ~0.7407)')
print(f'ET  OOF AUC: {auc_et:.5f}')
print()
print(f'{"조합":<30} {"probability min":>16} {"probability max":>16}')
print('-' * 65)
for label, pred in combos.items():
    print(f'  {label:<28} {pred.min():.4f}          {pred.max():.4f}')

RF  OOF AUC: 0.73631  (참고: GBDT 기준선 ~0.7407)
ET  OOF AUC: 0.73575

조합                              probability min  probability max
-----------------------------------------------------------------
  RF만                          0.0000          1.0000
  RF+ET                        0.0000          1.0000
  020+028+032 (기존)             0.0000          1.0000
  020+028+032+RF               0.0000          1.0000
  020+028+032+ET               0.0000          1.0000
  020+028+032+RF+ET            0.0000          1.0000
  020+032+RF                   0.0000          1.0000
  028+032+RF                   0.0000          1.0000


## 5. Submission 저장

상관관계가 낮을수록 앙상블 효과가 크므로, RF/ET의 GBDT 상관관계를 보고 포함 여부 결정.

In [6]:
SUB_DIR.mkdir(parents=True, exist_ok=True)

# 상관관계 분석 기반으로 최적 조합 선택
# RF/ET ↔ GBDT 상관관계가 0.96 미만이면 포함 효과 있음

save_combos = [
    ('020+028+032+RF',    combos['020+028+032+RF']),
    ('020+028+032+ET',    combos['020+028+032+ET']),
    ('020+028+032+RF+ET', combos['020+028+032+RF+ET']),
    ('020+032+RF',        combos['020+032+RF']),
]

saved = []
for label, pred in save_combos:
    fname = f'submission_exp{EXP_NO:03d}_{AUTHOR}_{label.replace("+","_")}.csv'
    s = sub.copy()
    s['probability'] = pred
    s.to_csv(SUB_DIR / fname, index=False)
    saved.append(fname)
    print(f'저장: {fname}')

print(f'\n총 {len(saved)}개 파일 저장')
print(f'\n→ RF ↔ GBDT 상관관계가 0.90 미만이면 020+028+032+RF 우선 제출 추천')
print(f'→ RF ↔ GBDT 상관관계가 0.90~0.95 이면 020+028+032+RF+ET 추천')
print(f'→ RF ↔ GBDT 상관관계가 0.95 이상이면 기존 EXP-033(020+028+032) 유지')

저장: submission_exp035_조여진_020_028_032_RF.csv
저장: submission_exp035_조여진_020_028_032_ET.csv
저장: submission_exp035_조여진_020_028_032_RF_ET.csv
저장: submission_exp035_조여진_020_032_RF.csv

총 4개 파일 저장

→ RF ↔ GBDT 상관관계가 0.90 미만이면 020+028+032+RF 우선 제출 추천
→ RF ↔ GBDT 상관관계가 0.90~0.95 이면 020+028+032+RF+ET 추천
→ RF ↔ GBDT 상관관계가 0.95 이상이면 기존 EXP-033(020+028+032) 유지


## 6. 실험 기록

In [7]:
from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment, Border, Side, PatternFill

def log_to_leaderboard(exp_no, author, model_name, params_str,
                        f1, recall, precision, accuracy, oof_auc,
                        cv_strategy, preprocessing_ver, n_features,
                        imbalance_method, submitted, hackathon_score,
                        file_name, notes='', insights=''):
    lb_path = DOCS_DIR / 'leaderboard.xlsx'
    wb = load_workbook(lb_path)
    ws = wb['리더보드']
    exp_label = f'EXP-{exp_no:03d}'
    next_row = ws.max_row + 1
    for r in range(2, ws.max_row + 1):
        val = ws.cell(row=r, column=2).value
        if val == exp_label:
            next_row = r; break
        if ws.cell(row=r, column=1).value is None or str(ws.cell(row=r, column=1).value).strip() == '':
            next_row = r; break
    values = [str(date.today()), exp_label, author, model_name, params_str,
              round(f1,5) if f1 else None, round(recall,5) if recall else None,
              round(precision,5) if precision else None, round(accuracy,5) if accuracy else None,
              round(oof_auc,5),
              cv_strategy, preprocessing_ver, n_features, imbalance_method,
              submitted, hackathon_score, file_name, notes, insights]
    thin   = Side(style='thin', color='B0B8D0')
    border = Border(left=thin, right=thin, top=thin, bottom=thin)
    fill   = PatternFill('solid', fgColor='EEF2FA') if next_row % 2 == 0 else None
    font   = Font(name='맑은 고딕', size=10)
    center = Alignment(horizontal='center', vertical='center', wrap_text=True)
    left   = Alignment(horizontal='left',   vertical='center', wrap_text=True)
    left_cols = {4, 5, 12, 14, 17, 18, 19}
    for c_idx, val in enumerate(values, start=1):
        cell = ws.cell(row=next_row, column=c_idx, value=val)
        cell.font = font; cell.border = border
        cell.alignment = left if c_idx in left_cols else center
        if fill: cell.fill = fill
        if c_idx in range(6, 11) or c_idx == 16: cell.number_format = '0.00000'
    wb.save(lb_path)
    print(f'[leaderboard.xlsx] EXP-{exp_no:03d} 기록 완료 (row {next_row})')

oof_binary_rf = (oof_rf >= 0.5).astype(int)
params_str = (f'RF(n=500,leaf=20,balanced)+ET(n=500,leaf=20,balanced) | '
              f'앙상블: RF+ET+EXP-020+EXP-028+EXP-032 Rank Avg')
NOTES    = ('RF/ET 단독 OOF AUC 기록. 앙상블은 기존 submission 파일과 test 예측 결합. '
            'RF ↔ GBDT 상관관계 낮으면 앙상블 효과 큼.')
INSIGHTS = (f'RF={auc_rf:.5f}  ET={auc_et:.5f}  RF+ET={auc_rf_et:.5f} | '
            f'GBDT 기준선 0.74068 대비 {auc_rf-0.74068:+.5f} (RF 단독)')

log_to_leaderboard(
    EXP_NO, AUTHOR, 'RF+ET Ensemble', params_str,
    f1_score(y_train, oof_binary_rf),
    recall_score(y_train, oof_binary_rf),
    precision_score(y_train, oof_binary_rf),
    accuracy_score(y_train, oof_binary_rf),
    auc_rf, CV_STR, 'v1', X_train.shape[1],
    'class_weight=balanced',
    'N', None, 'notebooks/31_rf_ensemble_yjcho.ipynb', NOTES, INSIGHTS
)

[leaderboard.xlsx] EXP-035 기록 완료 (row 32)
